# Coffee17 HF–Deep Complementarity V1 — Kaggle

Registered screening: HF20 vs HBP-EMB vs HF20+HBP-EMB. Outer test remains locked.

In [ ]:
NOTEBOOK_BUILD = "COFFEE17-HF-DEEP-COMPLEMENTARITY-V1"
SCIENTIFIC_CODE_COMMIT = "ac6eceee8319a6fd970d9f1de955bf300d3be115"

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import hashlib, importlib, json, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification-hf-deep"
PROJECT = WORK / "coffee17-hf-deep-complementarity-v1-project"
OUT = PROJECT / "experiments/coffee17-hf-deep-complementarity-v1"
CACHE = PROJECT / "cache/tulsi71_shared.npz"
RESUME_ZIP = WORK / "coffee17-hf-deep-complementarity-v1-resume-package.zip"

assert INPUT.is_dir() and WORK.is_dir(), "Notebook ini harus dijalankan di Kaggle."

def write_resume_archive():
    if not PROJECT.exists():
        return None
    if RESUME_ZIP.exists():
        RESUME_ZIP.unlink()
    archive = Path(shutil.make_archive(
        str(RESUME_ZIP.with_suffix("")),
        "zip",
        root_dir=PROJECT,
    ))
    print("\nRESUME PACKAGE:", archive, flush=True)
    return archive

def run(command, cwd=None, log_path=None):
    command = [str(x) for x in command]
    print("\n$ " + " ".join(command), flush=True)
    env = os.environ.copy()
    if log_path is None:
        subprocess.run(command, cwd=cwd, check=True, env=env)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(
            command, cwd=cwd, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1, env=env,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
            stream.flush()
        rc = process.wait()
    if rc:
        write_resume_archive()
        raise RuntimeError(f"Command gagal ({rc}): {' '.join(command)}")

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# Restore this experiment only. Do not mix it with DCL outputs.
prior_projects = sorted(
    p for p in INPUT.rglob("coffee17-hf-deep-complementarity-v1-project")
    if p.is_dir()
)
resume_archives = sorted(
    p for p in INPUT.rglob("coffee17-hf-deep-complementarity-v1-resume-package.zip")
    if p.is_file()
)
if len(prior_projects) > 1 or len(resume_archives) > 1:
    raise RuntimeError("Resume source HF-DEEP ambigu.")
if prior_projects and resume_archives:
    raise RuntimeError("Pilih satu resume source: project directory ATAU resume ZIP.")
if prior_projects:
    print("RESTORE PRIOR PROJECT:", prior_projects[0])
    shutil.rmtree(PROJECT, ignore_errors=True)
    shutil.copytree(prior_projects[0], PROJECT)
elif resume_archives:
    print("RESTORE RESUME ZIP:", resume_archives[0])
    shutil.rmtree(PROJECT, ignore_errors=True)
    PROJECT.mkdir(parents=True)
    shutil.unpack_archive(resume_archives[0], PROJECT)

PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT / "logs").mkdir(parents=True, exist_ok=True)
(PROJECT / "analysis").mkdir(parents=True, exist_ok=True)
(PROJECT / "cache").mkdir(parents=True, exist_ok=True)
(PROJECT / "imported_hbp").mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

if REPO.exists():
    shutil.rmtree(REPO)
run([
    "git", "clone", "--quiet", "--no-checkout",
    "https://github.com/ediprin/coffee-bean-classification.git", REPO
])
run(["git", "-C", REPO, "checkout", "--quiet", "--detach", SCIENTIFIC_CODE_COMMIT])

checked_out = subprocess.check_output(
    ["git", "-C", REPO, "rev-parse", "HEAD"], text=True
).strip()
if checked_out != SCIENTIFIC_CODE_COMMIT:
    raise RuntimeError(f"Checkout salah: {checked_out} != {SCIENTIFIC_CODE_COMMIT}")

runner_text = (
    REPO / "src/bilinear_lmmd/experiments/run_hf_deep_complementarity.py"
).read_text(encoding="utf-8")
feature_text = (
    REPO / "src/bilinear_lmmd/features/tulsi_handcrafted.py"
).read_text(encoding="utf-8")
for marker in (
    "coffee17-hf-deep-complementarity-v1",
    "def hbp_gpu_smoke_test",
    "def _shared_hf_features",
):
    if marker not in runner_text:
        raise RuntimeError(f"Runner marker hilang: {marker}")
for marker in ("FEATURE_NAMES", "def extract_tulsi71", "def mrmr_select"):
    if marker not in feature_text:
        raise RuntimeError(f"Feature marker hilang: {marker}")

run([
    sys.executable, "-m", "pip", "install", "-q", "-r",
    REPO / "requirements/preprocessing-study.txt"
])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan Kaggle GPU sebelum Run All.")

print("NOTEBOOK_BUILD:", NOTEBOOK_BUILD)
print("SCIENTIFIC_CODE_COMMIT:", SCIENTIFIC_CODE_COMMIT)
print("GPU:", torch.cuda.get_device_name(0))

run([
    sys.executable, "-m", "pytest", "-q",
    "tests/features/test_tulsi_handcrafted.py",
    "tests/experiments/test_hf_deep_complementarity.py",
], cwd=REPO)
print("PRE-TRAIN TESTS: PASS")

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import (
    materialize_preprocessing_development,
)
from bilinear_lmmd.engine.physical_logit_residual_wr_hbp import configure_strict_determinism

det = configure_strict_determinism(42)
print("\n=== STRICT DETERMINISM ===")
print(json.dumps(det, indent=2))

by_class = discover_directory_samples(INPUT)
raw_count = sum(len(v) for v in by_class.values())
print("Coffee17 mounted:", raw_count, "images /", len(by_class), "classes")
if raw_count != 979 or len(by_class) != 17:
    raise RuntimeError(
        "Input harus Coffee17 original: "
        f"expected 979 images / 17 classes, observed {raw_count}/{len(by_class)}."
    )

ARCHIVE = WORK / "coffee17_hfdeep_original.zip"
PROV_LOCAL = WORK / "coffee17_hfdeep_provenance"
CANONICAL = WORK / "coffee17_hfdeep_original_v1"
FOLDS_LOCAL = WORK / "coffee17_hfdeep_folds"

for path in (PROV_LOCAL, CANONICAL, FOLDS_LOCAL):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info = zipfile.ZipInfo(
                f"{class_name}/{path.name}",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            info.compress_type = zipfile.ZIP_STORED
            info.external_attr = 0o644 << 16
            bundle.writestr(info, path.read_bytes())

provenance = audit_coffee17_provenance(
    ARCHIVE, PROV_LOCAL, canonical_root=CANONICAL
)
if provenance["decision"] != "PASS":
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary = prepare_preprocessing_folds(
    CANONICAL,
    PROV_LOCAL / "coffee17_provenance.json",
    FOLDS_LOCAL,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary["decision"] != "PASS_COFFEE17_PREPROCESSING_DATA_GATE":
    raise RuntimeError("Fold reconstruction gagal.")

CONFIG = REPO / "configs/hf_deep_complementarity/HF_DEEP_COMPLEMENTARITY_V1.yaml"

def external_hbp_checkpoint(fold):
    # Reuse is optional. Only exact HBP_CE best checkpoints from a prior DCL
    # run are eligible; otherwise this experiment trains its own matched HBP.
    local = PROJECT / "imported_hbp" / f"fold_{fold}_HBP_CE_best.pt"
    if local.is_file():
        return local

    candidates = []
    for p in INPUT.rglob("best.pt"):
        parts = set(p.parts)
        if "HBP_CE" in parts and f"fold_{fold}" in parts:
            candidates.append(p)
    candidates = sorted(candidates)
    if not candidates:
        return None

    hashes = {file_sha256(p) for p in candidates}
    if len(hashes) != 1:
        raise RuntimeError(
            f"Fold {fold}: lebih dari satu HBP_CE checkpoint berbeda ditemukan."
        )
    print(f"IMPORT PRIOR HBP fold {fold}: {candidates[0]}")
    shutil.copy2(candidates[0], local)
    return local

for fold in range(1, 6):
    print(f"\n================ FOLD {fold}/5 ================", flush=True)
    DEV = WORK / f"coffee17_hfdeep_dev_fold_{fold}"
    shutil.rmtree(DEV, ignore_errors=True)

    materialize_preprocessing_development(
        CANONICAL,
        FOLDS_LOCAL / "clean_manifest.json",
        FOLDS_LOCAL / "fold_manifest.json",
        DEV,
        fold=fold,
    )

    checkpoint = external_hbp_checkpoint(fold)
    cmd = [
        sys.executable, "-u", "-m",
        "bilinear_lmmd.experiments.run_hf_deep_complementarity",
        "--config", CONFIG,
        "--fold", str(fold),
        "--data-root", DEV,
        "--output-root", OUT,
        "--required-commit", SCIENTIFIC_CODE_COMMIT,
        "--device", "cuda:0",
        "--resume",
        "--authorize-training",
        "--shared-hf-cache", CACHE,
    ]
    if checkpoint is not None:
        cmd.extend(["--hbp-checkpoint", checkpoint])

    run(cmd, cwd=REPO, log_path=PROJECT / "logs" / f"fold_{fold}.log")

    result = json.loads(
        (OUT / f"fold_{fold}" / "seed42" / "fold_result.json").read_text()
    )
    d = result["delta_fusion_minus_deep"]
    print(
        f"FOLD {fold} | "
        f"HF20={result['arms']['HF20']['macro_f1']:.4f} | "
        f"HBP={result['arms']['HBP_EMB']['macro_f1']:.4f} | "
        f"FUSION={result['arms']['HF20_HBP_EMB']['macro_f1']:.4f} | "
        f"Delta Macro={d['macro_f1']:+.4f} | "
        f"Delta Hard={d['hard_class_f1']:+.4f} | "
        f"Delta Worst={d['worst_class_f1']:+.4f} | "
        f"Rescue/Damage={result['paired_outcomes_fusion_vs_deep']['rescue']}/"
        f"{result['paired_outcomes_fusion_vs_deep']['damage']} | "
        f"PairDelta={result['targeted_confusion_delta_fusion_minus_deep']['TOTAL']:+d}"
    )

    shutil.rmtree(DEV, ignore_errors=True)
    torch.cuda.empty_cache()

SUMMARY = PROJECT / "analysis" / "hf_deep_complementarity_summary.json"
run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.run_hf_deep_complementarity_summary",
    "--output-root", OUT,
    "--config", CONFIG,
    "--output", SUMMARY,
], cwd=REPO)

summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
delta = summary["delta_fusion_minus_deep"]
gate = summary["screening_gate"]

print("\n=== DECISION ===")
print("Gate:", gate["decision"])
print(
    "Macro delta:", f"{delta['macro_f1']['mean']:+.4%}",
    "| positive folds:", delta["macro_f1"]["positive_folds"], "/5",
)
print("Hard delta:", f"{delta['hard_class_f1']['mean']:+.4%}")
print("Worst delta:", f"{delta['worst_class_f1']['mean']:+.4%}")
print(
    "Hard-pair confusions:",
    summary["targeted_confusions_total"]["HBP_EMB"],
    "->",
    summary["targeted_confusions_total"]["HF20_HBP_EMB"],
)
print(
    "Rescue / damage:",
    summary["paired_prediction_outcomes"]["rescue"],
    "/",
    summary["paired_prediction_outcomes"]["damage"],
)
print("Outer test accessed:", summary["outer_test_accessed"])

PKG = WORK / "coffee17-hf-deep-complementarity-v1-analysis-package"
shutil.rmtree(PKG, ignore_errors=True)
PKG.mkdir(parents=True)
shutil.copy2(SUMMARY, PKG / SUMMARY.name)
shutil.copy2(CONFIG, PKG / CONFIG.name)
shutil.copy2(
    REPO / "docs/protocols/COFFEE17_HF_DEEP_COMPLEMENTARITY_V1.md",
    PKG / "COFFEE17_HF_DEEP_COMPLEMENTARITY_V1.md",
)

for fold in range(1, 6):
    src = OUT / f"fold_{fold}" / "seed42"
    dst = PKG / f"fold_{fold}"
    dst.mkdir(parents=True)
    for name in ("fold_result.json", "feature_selection.json"):
        shutil.copy2(src / name, dst / name)
    for arm in ("HF20", "HBP_EMB", "HF20_HBP_EMB"):
        arm_src = src / arm
        arm_dst = dst / arm
        arm_dst.mkdir(parents=True)
        for name in ("metrics.json", "predictions.csv", "probe.npz"):
            p = arm_src / name
            if p.is_file():
                shutil.copy2(p, arm_dst / name)
    hbp_src = src / "hbp_core"
    if hbp_src.is_dir():
        hbp_dst = dst / "hbp_core"
        hbp_dst.mkdir(parents=True)
        for name in ("run_contract.json", "history.json"):
            p = hbp_src / name
            if p.is_file():
                shutil.copy2(p, hbp_dst / name)

zip_path = Path(shutil.make_archive(
    str(WORK / "coffee17-hf-deep-complementarity-v1-analysis-package"),
    "zip",
    root_dir=PKG,
))
print("\nREADY:", zip_path)
print("SIZE:", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")

if RESUME_ZIP.exists():
    RESUME_ZIP.unlink()

for path in (REPO, PROV_LOCAL, CANONICAL, FOLDS_LOCAL, PKG):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()
